In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, roc_curve
from scipy.stats import ks_2samp
from imblearn.over_sampling import RandomOverSampler

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D, Reshape, TimeDistributed
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

import optuna

# Pasta dos dados escalados e coluna alvo
scaled_data_folder = "scaled_data"
target_col = "Churn"

def kolmogorov_smirnov(y_true, y_prob):
    y_bin = np.array(y_true)
    pos_probs = y_prob[y_bin == 1]
    neg_probs = y_prob[y_bin == 0]
    ks_stat, p_value = ks_2samp(pos_probs, neg_probs)
    return ks_stat, p_value

def evaluate_metrics(y_true, y_prob, threshold=0.5):
    y_bin = np.array(y_true)
    y_pred = (y_prob >= threshold).astype(int)
    
    cm = confusion_matrix(y_bin, y_pred)
    precision = precision_score(y_bin, y_pred)
    recall = recall_score(y_bin, y_pred)
    f1 = f1_score(y_bin, y_pred)
    auc = roc_auc_score(y_bin, y_prob)
    mse = mean_squared_error(y_bin, y_prob)
    ks_stat, ks_p = kolmogorov_smirnov(y_true, y_prob)
    
    return {
        "cm": cm,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "mse": mse,
        "ks_stat": ks_stat,
        "ks_p": ks_p
    }

def plot_ks(y_true, y_score):
    """
    Gera a curva de KS (Kolmogorov-Smirnov) plotando:
      - Eixo X: Score
      - Eixo Y: Probabilidade acumulada
      - Duas curvas: CDF de eventos (positivos) e CDF de não-eventos (negativos)
      - Linha vertical mostrando o ponto com a maior diferença KS
    """
    y_true = np.array(y_true).astype(int)
    y_score = np.array(y_score)
    
    order = np.argsort(y_score)
    y_score_sorted = y_score[order]
    y_true_sorted = y_true[order]
    
    total_pos = np.sum(y_true == 1)
    total_neg = np.sum(y_true == 0)
    
    cum_pos = np.cumsum(y_true_sorted) / total_pos
    cum_neg = np.cumsum(1 - y_true_sorted) / total_neg
    
    ks_values = cum_pos - cum_neg
    ks_max = np.max(np.abs(ks_values))
    idx_ks = np.argmax(np.abs(ks_values))
    best_threshold = y_score_sorted[idx_ks]
    
    plt.figure(figsize=(8, 5))
    plt.plot(y_score_sorted, cum_pos, label='CDF de Positivos', color='red')
    plt.plot(y_score_sorted, cum_neg, label='CDF de Negativos', color='blue')
    plt.vlines(x=best_threshold,
               ymin=min(cum_pos[idx_ks], cum_neg[idx_ks]),
               ymax=max(cum_pos[idx_ks], cum_neg[idx_ks]),
               colors='k', linestyles='--',
               label=f'KS={ks_max:.2%} at score={best_threshold:.2f}')
    plt.xlabel('Score')
    plt.ylabel('Probabilidade Acumulada')
    plt.title('Curva KS (CDF)')
    plt.legend()
    plt.show()
    
    print(f"KS = {ks_max:.4f} (ou {ks_max:.2%}) no threshold = {best_threshold:.4f}")

def transformer_encoder_block(inputs, num_heads, ff_dim, dropout_rate):
    # Bloco Transformer padrão: atenção multi-head, dropout, normalização e feed-forward
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=ff_dim)(inputs, inputs)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attn_output)
    
    ffn_output = Dense(ff_dim, activation='relu')(out1)
    ffn_output = Dense(inputs.shape[-1])(ffn_output)
    ffn_output = Dropout(dropout_rate)(ffn_output)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ffn_output)
    return out2

def build_tabkanet(input_dim, num_kan_layers, kan_units, num_transformer_blocks, num_heads, ff_dim, dropout_rate, final_units, learning_rate):
    """
    Implementação do TabKANet:
      - Cada feature é tratada como um token (através do Reshape).
      - Um bloco inspirado em KAN é aplicado via TimeDistributed, processando cada feature individualmente.
      - Em seguida, os tokens são projetados para uma dimensão adequada para o Transformer.
      - Aplica-se uma sequência de blocos Transformer para capturar interações entre as features.
      - Os tokens são agregados (via GlobalAveragePooling1D) e passam por camadas densas para a classificação.
    """
    inputs = Input(shape=(input_dim,))
    # Reshape para tratar cada feature como um token: (batch_size, num_features, 1)
    x = Reshape((input_dim, 1))(inputs)
    
    # Bloco KAN: processamento individual das features usando TimeDistributed
    for _ in range(num_kan_layers):
        x = TimeDistributed(Dense(kan_units, activation='relu'))(x)
        x = TimeDistributed(Dropout(dropout_rate))(x)
    
    # Projeção para o espaço do Transformer
    x = TimeDistributed(Dense(ff_dim))(x)
    
    # Blocos Transformer
    for _ in range(num_transformer_blocks):
        x = transformer_encoder_block(x, num_heads=num_heads, ff_dim=ff_dim, dropout_rate=dropout_rate)
    
    # Agregação dos tokens
    x = GlobalAveragePooling1D()(x)
    # Camada densa final para aprendizado
    x = Dense(final_units, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# Loop para rodar nos 3 folds
all_fold_results = []
n_folds = 3

for i in range(n_folds):
    print(f"\n===== FOLD {i+1} =====")
    
    # Carregar CSVs
    train_df = pd.read_csv(os.path.join(scaled_data_folder, f"train_fold_{i+1}_scaled.csv"))
    val_df = pd.read_csv(os.path.join(scaled_data_folder, f"val_fold_{i+1}_scaled.csv"))
    test_df = pd.read_csv(os.path.join(scaled_data_folder, f"test_fold_{i+1}_scaled.csv"))
    
    # Separar features e target para treino
    X_train = train_df.drop(columns=[target_col]).values
    y_train = np.where(train_df[target_col] == 'Yes', 1, 0)
    
    # Separar features e target para validação e teste
    X_val = val_df.drop(columns=[target_col]).values
    y_val = np.where(val_df[target_col] == 'Yes', 1, 0)
    X_test = test_df.drop(columns=[target_col]).values
    y_test = np.where(test_df[target_col] == 'Yes', 1, 0)
    
    # Oversampling para balancear o conjunto de treinamento
    ros = RandomOverSampler(random_state=42)
    X_train_res, y_train_res = ros.fit_resample(X_train, y_train)
    
    # Otimização de hiperparâmetros com Optuna para o TabKANet
    def objective(trial):
        num_kan_layers = trial.suggest_int("num_kan_layers", 1, 3)
        kan_units = trial.suggest_int("kan_units", 16, 64, step=16)
        num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 3)
        num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
        ff_dim = trial.suggest_int("ff_dim", 16, 64, step=16)
        dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
        final_units = trial.suggest_int("final_units", 32, 128, step=16)
        learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        
        model = build_tabkanet(X_train.shape[1],
                               num_kan_layers=num_kan_layers,
                               kan_units=kan_units,
                               num_transformer_blocks=num_transformer_blocks,
                               num_heads=num_heads,
                               ff_dim=ff_dim,
                               dropout_rate=dropout_rate,
                               final_units=final_units,
                               learning_rate=learning_rate)
        
        history = model.fit(
            X_train_res, y_train_res,
            validation_data=(X_val, y_val),
            epochs=15, batch_size=32, verbose=0
        )
        
        y_val_probs = model.predict(X_val).flatten()
        ks_stat, _ = kolmogorov_smirnov(y_val, y_val_probs)
        return ks_stat

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50)
    
    best_params = study.best_params
    print(f"Melhores parâmetros do Optuna para TabKANet: {best_params}")
    
    best_model = build_tabkanet(X_train.shape[1],
                                num_kan_layers=best_params["num_kan_layers"],
                                kan_units=best_params["kan_units"],
                                num_transformer_blocks=best_params["num_transformer_blocks"],
                                num_heads=best_params["num_heads"],
                                ff_dim=best_params["ff_dim"],
                                dropout_rate=best_params["dropout_rate"],
                                final_units=best_params["final_units"],
                                learning_rate=best_params["learning_rate"])
    
    history = best_model.fit(
        X_train_res, y_train_res,
        validation_data=(X_val, y_val),
        epochs=50, batch_size=32, verbose=0
    )
    
    # Avaliação no conjunto de teste
    y_test_probs = best_model.predict(X_test).flatten()
    test_metrics = evaluate_metrics(y_test, y_test_probs)
    
    print("\n-- Métricas de Teste --")
    print(f"Confusion Matrix:\n{test_metrics['cm']}")
    print(f"Precision: {test_metrics['precision']:.4f} | Recall: {test_metrics['recall']:.4f} | F1: {test_metrics['f1']:.4f}")
    print(f"AUC: {test_metrics['auc']:.4f} | MSE: {test_metrics['mse']:.4f} | KS: {test_metrics['ks_stat']:.4f}")
    
    all_fold_results.append(test_metrics)
    
    # Plotando a Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_test_probs)
    plt.figure(figsize=(8, 5))
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC={test_metrics['auc']:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Fold {i+1} - Curva ROC")
    plt.legend()
    plt.show()
    
    # Plotando a Curva KS (CDF)
    plot_ks(y_test, y_test_probs)
